# Hermes Colab 节点自检 v2
GPU金标准=nvidia-smi, 输出机器可解析marker

In [ ]:
import subprocess, json, sys, os, platform
result = {
    'python': sys.version.split()[0],
    'machine': platform.machine(),
    'cpu': os.cpu_count(),
    'cuda_env': os.environ.get('CUDA_VISIBLE_DEVICES'),
    'gpu': []
}
p = subprocess.run(['nvidia-smi','--query-gpu=name,driver_version,memory.total','--format=csv,noheader,nounits'], capture_output=True, text=True)
if p.returncode == 0 and p.stdout.strip():
    for line in p.stdout.strip().splitlines():
        parts = [x.strip() for x in line.split(',')]
        result['gpu'].append({'name': parts[0], 'driver': parts[1], 'mem_mb': int(parts[2])})
mem = os.popen('free -g').read().splitlines()
if len(mem) > 1: result['ram_gb'] = mem[1].split()[1]
dsk = os.popen('df -h /content 2>/dev/null | tail -1 || df -h / | tail -1').read().split()
result['disk_free'] = dsk[3] if len(dsk) > 3 else dsk[-4] if len(dsk)>4 else '?'
import urllib.request
try:
    result['net'] = urllib.request.urlopen('https://example.com', timeout=8).status
except Exception as e:
    result['net'] = str(e)[:50]
print('COLAB_GPU_RESULT=' + json.dumps([g['name'] for g in result['gpu']], ensure_ascii=False))
print('COLAB_PROBE=' + json.dumps(result, ensure_ascii=False))
print('COLAB_NODE_V2_DONE')